# Creación de Sistemas de Recomendación para Películas

En este notebook, desarrollaremos tres sistemas de recomendación para ayudar a los usuarios a descubrir películas según sus preferencias. Los sistemas que implementaremos son:

**Sistema de recomendación basado en popularidad**: Este sistema se basa en recomendar las películas más populares, es el sistema más simple.

**Sistema de recomendación basado en contenido:** Este sistema recomienda películas que son similares a las que un usuario ha visto previamente, basándose en características como el género, keywords o el director.

**Sistema de recomendación colabrativo**: : Este sistema utiliza las valoraciones y las interacciones previas de los usuarios para recomendar películas similares.

Para los dos primeros modelos, utilizaremos el dataset de Kaggle TMDB_5000, que contiene información de 5,000 películas, incluyendo características como género, actores, presupuesto, entre otras.

Dado que ya hemos procesado este dataset en el notebook "01_EDA.ipynb", lo cargaremos ya limpio en este notebook para su uso:

In [1]:
import pandas as pd
df = pd.read_csv("../data/cleandf.csv")
df.head()

,movie_id,title,cast,crew,budget,genres,keywords,original_language,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,vote_average,vote_count,release_year
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...",150.437577,"['Ingenious Film Partners', 'Twentieth Century...","['United States of America', 'United Kingdom']",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",7.2,11800,2009.0
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"['Adventure', 'Fantasy', 'Action']","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...",139.082615,"['Walt Disney Pictures', 'Jerry Bruckheimer Fi...",['United States of America'],2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",6.9,4500,2007.0
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",245000000,"['Action', 'Adventure', 'Crime']","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,A cryptic message from Bond’s past sends him o...,107.376788,"['Columbia Pictures', 'Danjaq', 'B24']","['United Kingdom', 'United States of America']",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",6.3,4466,2015.0
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",250000000,"['Action', 'Crime', 'Drama', 'Thriller']","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,Following the death of District Attorney Harve...,112.312950,"['Legendary Pictures', 'Warner Bros.', 'DC Ent...",['United States of America'],2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",7.6,9106,2012.0
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",260000000,"['Action', 'Adventure', 'Science Fiction']","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,"John Carter is a war-weary, former military ca...",43.926995,['Walt Disney Pictures'],['United States of America'],2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",6.1,2124,2012.0


## Sistema basado en popularidad/demografía

Inspirándonos en los sistemas que usa netflix, este sistema se podría equiparar a:

![Trends](../images/trendmovies.png)


Donde se muestran las películas con más popularidad en tu país en el momento actual o durante un periodo de tiempo.

Para crear este sistema necesitamos:

- Una métrica para puntuar o calificar una película

- Ordenar las puntuaciones

- Recomendar la película mejor valorada a los usuarios

Podemos usar las valoraciones promedio de la película como la puntuación, pero usar solo esto no sería justo, ya que una película con una valoración promedio de 9 y solo 7 votos no puede considerarse mejor que una película con una valoración promedio de 7.8 pero con 100 votos. Por lo tanto, utilizaré la clasificación ponderada (weighted rating, wr) de IMDB, que se da de la siguiente manera:

<img src="../images/wrformula.png" alt="WRformula" width="750"/>


Ya tenemeos en el dataset V(cote_count) y R(cote_average), además, podemos calcular la C fácilmente:

In [2]:
C = df['vote_average'].mean()
C

np.float64(6.092171559442016)


La calificación media de todas las películas es aproximadamente 6 en una escala de 10. El siguiente paso es determinar un valor adecuado para m, el número mínimo de votos requeridos para aparecer en la lista. Este número es más subjetivo, pero usaremos el percentil 90 como nuestro umbral, es decir, para que una película aparezca en la lista, debe tener más votos que al menos el 90% de las películas en la lista.

In [3]:
m= df['vote_count'].quantile(0.9)
m

np.float64(1838.4000000000015)

Ahora, filtramos las películas que entran en este rango:

In [4]:
top_movies = df.copy().loc[df['vote_count'] >= m]
top_movies.shape

(481, 19)

Tenemos 481 películas de alrededor de 5.000 que entran en posibles recomendaciones, ahora aplicaremos al fórmula WR para darles su nueva puntuación ponderada.

In [5]:
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

Y creamos una nueva columna con esta puntuación

In [6]:
top_movies['score'] = top_movies.apply(weighted_rating, axis=1)

Ahora ordenamos las películas según esta puntuación y vemos los resultados:

In [7]:
top_movies = top_movies.sort_values('score', ascending=False)

top_movies[['title', 'vote_count', 'vote_average', 'score']].head(10)

,title,vote_count,vote_average,score
1881,The Shawshank Redemption,8205,8.5,8.059258
662,Fight Club,9413,8.3,7.939256
65,The Dark Knight,12002,8.2,7.920020
3232,Pulp Fiction,8428,8.3,7.904645
96,Inception,13752,8.1,7.863239
3337,The Godfather,5893,8.4,7.851236
95,Interstellar,10867,8.1,7.809479
809,Forrest Gump,7927,8.2,7.803188
329,The Lord of the Rings: The Return of the King,8064,8.1,7.727243
1990,The Empire Strikes Back,5879,8.2,7.697884


Y con esto ya tendríamos nuestro sistema de recomendación listo.

Este sistema de recomendación basado en popularidad calcula el puntaje Weighted Rating (WR) de las películas a nivel global. Sin embargo, para personalizar las recomendaciones según la ubicación del usuario, sería necesario segmentar las películas por país o región del usuario utilizando su ubicación geográfica. Así, las recomendaciones se adaptan mejor a las preferencias locales y culturales de cada usuario. También se podría mejorar incluyendo un período de tiempo diario, semanal o mensual.


Este sistema, anque puede ser bastante útil, no es sensibles a los intereses y gustos específicos de cada usuario en particular. 

Es aquí cuando pasamos a un sistema más elaborado: el Filtrado Basado en Contenido.

## Sistema de Filtrado Basado en Contenido

El sistema de filtrado basado en contenido recomienda películas en función de las características similares a las que el usuario ya ha visto o calificado positivamente. Utiliza atributos como género, director, actores, palabras clave y descripción para identificar similitudes entre películas. A diferencia del sistema basado en popularidad, este enfoque toma en cuenta el contenido específico de las películas, lo que lo hace más personalizado. 

Volviendo al ejemplo de **Netflix**, sería equiparable a:

![NetflixContentBased](../images/contentbased.png)


Empezaremos calculando las puntuaciones según su descripción (overview) y haremos recomendaciones en base a esa puntuación.

In [8]:
df.overview

0       In the 22nd century, a paraplegic Marine is di...
1       Captain Barbossa, long believed to be dead, ha...
2       A cryptic message from Bond’s past sends him o...
3       Following the death of District Attorney Harve...
4       John Carter is a war-weary, former military ca...
                              ...                        
4798    El Mariachi just wants to play his guitar and ...
4799    A newlywed couple's honeymoon is upended by th...
4800    "Signed, Sealed, Delivered" introduces a dedic...
4801    When ambitious New York attorney Sam is sent t...
4802    Ever since the second grade when he first saw ...
Name: overview, Length: 4803, dtype: object

Convertiremos estas descripciones, que son cadenas de texto (strings), a vectores numéricos para poder calcular la similitud entre las películas. Para ello, utilizaremos la técnica **TF-IDF (Term Frequency-Inverse Document Frequency)**, que asigna un valor numérico a cada palabra en función de su frecuencia en una película y su rareza en el conjunto total de películas. Esto nos permitirá comparar las descripciones de las películas y determinar cuáles son más similares entre sí.

In [9]:
#Importamos TfIdfVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer

#Como el dataset está en ingles, eliminamos las stopwords del ingles
tfidf = TfidfVectorizer(stop_words='english')

#Eliminamos nulos
df['overview'] = df['overview'].fillna('')

#Construimos la matriz TF-IDF
tfidf_matrix = tfidf.fit_transform(df['overview'])

#Output the shape of tfidf_matrix
tfidf_matrix.shape

(4803, 20978)

Podemos ver que hay un total de 20978 palabras en las descripciones de 4803 películas

Una vez tenemos la matriz, podemos calcular la puntuación, en este caso utilizaremos la similaridad del coseno ya que es independiente de la magnitud y es fácil de calcular

In [10]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

Para hacerlo fácil, creamos una función que ingresando el título de una película, devuelva las 10 películas más similares. Para esto necesitamos indentificar el índice a partir del título.

En esta función obtendremos la lista, la ordenaremos según la similaridad del coseno, ignorando la primera, que es ella misma ya que tiene similaridad de 1 y devolveremos los títulos de las películas.

In [11]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [12]:
def get_recommendations(title, cosine_sim=cosine_sim):
  
    idx = indices[title]

    # Crea una lista de tuplas que contienen el índice y la puntuación de similitud entre la película dada y todas las demás.
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenamos la lista según similaridad
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Coge las 10 películas más similares, excluyendo la primera (que es la película en sí misma).
    sim_scores = sim_scores[1:11]

    # Extrae los índices de las películas similares.
    movie_indices = [i[0] for i in sim_scores]

    # Retorna las 10 más películas similares
    return df['title'].iloc[movie_indices]

Probemos con un ejemplo

In [13]:
df.title.sample(3)

3595    Halloween: The Curse of Michael Myers
3559                    Paranormal Activity 3
609                               Escape Plan
Name: title, dtype: object

In [14]:
get_recommendations('Batman Returns')

3                         The Dark Knight Rises
3854    Batman: The Dark Knight Returns, Part 2
65                              The Dark Knight
299                              Batman Forever
1359                                     Batman
119                               Batman Begins
9            Batman v Superman: Dawn of Justice
210                              Batman & Robin
1309                              Heartbreakers
504                     The Secret Life of Pets
Name: title, dtype: object

Podemos ver que tiene sentido, ya que devuelve continuaciones de esa misma Película, ya que la descripción tendrá bastantes elementos en común.

A pesar de eso, este sistema es muy simple y podríamos mejorar las recomendaciones incluyendo más elementos para calcular esta puntuación, de nuestro dataset. Sería interesante tener en cuenta el género, keywords, los protagonistas de las películas (actores) y el director. Podríamos incluir más variables como el idioma oficial o el año de estreno, pero habría que estudiar que solución es mejor para cada caso concreto, por ejemplo con un Test A/B.

De nuevo, variables como actores hayq eu estudiar cuantos tener en cuenta, para nuestro caso, consideramos 3 como un número razonable, con sentido y que hará el sistema rápido y simple.

Las variables genmres, keywords, cast y crew estaban en forma de stringfield list, aunque e el EDA hemos tratado Genres, debemos transformar el resto en listas tratables


In [15]:
df[['genres', 'keywords', 'cast', 'crew']]


,genres,keywords,cast,crew
0,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"['Adventure', 'Fantasy', 'Action']","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,"['Action', 'Adventure', 'Crime']","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,"['Action', 'Crime', 'Drama', 'Thriller']","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,"['Action', 'Adventure', 'Science Fiction']","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."
...,...,...,...,...
4798,"['Action', 'Crime', 'Thriller']","[{""id"": 5616, ""name"": ""united states\u2013mexi...","[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4799,"['Comedy', 'Romance']",[],"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4800,"['Comedy', 'Drama', 'Romance', 'TV Movie']","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de..."
4801,[],[],"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de..."


In [16]:
import ast

# Convertimos a objetos de Python reales
features = ['keywords', 'cast', 'crew']

for feature in features:
    df[feature] = df[feature].apply(ast.literal_eval)

In [17]:
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        # retornarmos 3 elementos
        if len(names) > 3:
            names = names[:3]
        return names

    return []

In [18]:
features = ['cast', 'keywords']
for feature in features:
    df[feature] = df[feature].apply(get_list)

In [19]:
import numpy as np

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [20]:
df['director'] = df['crew'].apply(get_director)

In [27]:
df[['title', 'cast', 'director', 'keywords', 'genres']].sample(5)

,title,cast,director,keywords,genres
3955,Just Looking,"[Gretchen Mol, Patti LuPone, Peter Onorati]",Jason Alexander,"[new york, nurse, sex]","['Drama', 'Comedy']"
171,Master and Commander: The Far Side of the World,"[Russell Crowe, Paul Bettany, James D'Arcy]",Peter Weir,"[naturalist, frigate, self surgery]",['Adventure']
2287,I Can Do Bad All By Myself,"[Tyler Perry, Taraji P. Henson, Adam Rodríguez]",Tyler Perry,"[aunt, duringcreditsstinger]","['Drama', 'Comedy']"
4686,Ordet,"[Birgitte Federspiel, Preben Lerdorff Rye, Hen...",Carl Theodor Dreyer,"[faith, independent film, religion]",['Drama']
4015,My Own Private Idaho,"[River Phoenix, Keanu Reeves, James Russo]",Gus Van Sant,"[individual, gay, father son relationship]","['Drama', 'Romance']"


Ahora lo convertiremos en minúsculas y eliminaremos espacios para que no haya fallos en el cálculo de la puntuación

In [28]:
def clean_data(x):
    if isinstance(x, list):  # Si x es una lista
        return [str.lower(i.replace(" ", "")) for i in x if isinstance(i, str)]  # Limpiar solo las cadenas de texto
    elif isinstance(x, str):  # Si x es un string
        return str.lower(x.replace(" ", ""))  # Limpiar el string
    else:
        return ''  # Si no es ni lista ni string, devolver vacío

# Aplicamos la función clean_data a las columnas correspondientes
values = ['cast', 'director', 'keywords', 'genres']
for elem in values:
    df[elem] = df[elem].apply(clean_data)


In [29]:
df[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[samworthington, zoesaldana, sigourneyweaver]",jamescameron,"[cultureclash, future, spacewar]","['action','adventure','fantasy','sciencefiction']"
1,Pirates of the Caribbean: At World's End,"[johnnydepp, orlandobloom, keiraknightley]",goreverbinski,"[ocean, drugabuse, exoticisland]","['adventure','fantasy','action']"
2,Spectre,"[danielcraig, christophwaltz, léaseydoux]",sammendes,"[spy, basedonnovel, secretagent]","['action','adventure','crime']"


Ahora crearemos "sopa de metadatos", que es una cadena de texto que contiene todos los metadatos que queremos alimentar a nuestro vectorizador (es decir, genero, actores, director y palabras clave).

Podemos darle más peso en el computo de la importancia a alguna de las variables como por ejemplo el género, pero aquí le daremos la misma a los 4 valores.

In [30]:
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])
df['soup'] = df.apply(create_soup, axis=1)

Ahora calcularemos la puntuación de las películas, peor utilizaremos CountVectorizer() en vez de TF-IDF, ya que no queremos disminuirla importancia de un valor si se repite mucho, que es loq ue hace TF-IDF.

In [31]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

In [33]:
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

Y ahora podemos utilizar de nuevo la función anterior para obtener recomendaciones, añadiendo cosine_sim2:

In [34]:
get_recommendations('The Dark Knight Rises', cosine_sim2)

65                         The Dark Knight
119                          Batman Begins
1196                          The Prestige
1246                     Quest for Camelot
1775                         The Statement
2460                            The Unborn
317                     The Flowers of War
2793                  The Killer Inside Me
3172                         The Contender
9       Batman v Superman: Dawn of Justice
Name: title, dtype: object

Podemos ver que ahora las recomendaciones han cambiado, ya que al capturar más datos, puede personalizar mucho mejor.

Con esto, hemos acabado nuestro sistema basado en contenido, podemos pasar al último.

## Sistema de Filtrado Colaborativo

Hasta el momento, nuestros recomendadores, a pesar de ser útiles, solo pueden captar películas similares a otras, y no gustos personales, no diferencian por cada persona, cualquiera que consulte nuestro motor para obtener recomendaciones basadas en una película recibirá las mismas recomendaciones para esa película, sin importar quién sea.

Trataremos de mejorar esto gracias a un sistema de filtrado colaborativo, que se centra en identificar patrones de comportamiento entre usuarios similares y utilizar esa información para recomendar ítems que otros usuarios con gustos similares han disfrutado


Volviendo de nuevo al ejemplo de netflix, esta parte correspondería a:

![NetflixColaborative](../images/colaborative.png)


Emplearemos un enfoque de Descomposición en Valores Singulares (SVD) para abordar los problemas de escalabilidad y dispersión del modelo. El SVD nos permite reducir la dimensión de la matriz de utilidad, extrayendo factores latentes que representan las características subyacentes de usuarios e ítems. Esto facilita una mejor comprensión de las relaciones entre ellos, haciendo que las predicciones de puntuación sean más precisas y efectivas.

Este modelo basado en filtrado colaborativo, junto con SVD, será nuestro último enfoque para ofrecer recomendaciones personalizadas y superar las limitaciones del sistema anterior. Con este nuevo modelo, esperamos proporcionar una experiencia personalizada a los usuarios.

Para este último ejemplo, utilizaremos otro dataset distinto, ya que necesitamos infromación de los usuarios y sus valoraciones, que en este caso, van del 1 al 5

In [44]:
from sklearn.decomposition import TruncatedSVD

# Cargar datos
ratings = pd.read_csv('../data/ratings_small.csv')

# Crear una tabla de usuario-producto con las valoraciones
ratings_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')

# Rellenar valores NaN con ceros
ratings_matrix.fillna(0, inplace=True)

# Realizar la descomposición SVD
svd = TruncatedSVD(n_components=50)  # Número de componentes que deseas conservar
latent_matrix = svd.fit_transform(ratings_matrix)

# Ver la matriz latente
print(latent_matrix.shape)


(671, 50)


In [45]:
# Reconstruir la matriz de valoraciones
predicted_ratings = np.dot(latent_matrix, svd.components_)

# Convertir en DataFrame para una visualización más fácil
predicted_ratings_df = pd.DataFrame(predicted_ratings, columns=ratings_matrix.columns)

# Ver las predicciones
print(predicted_ratings_df.head())


movieId    1         2         3         4         5         6         7       \
0       -0.041196  0.043325 -0.003182 -0.013017 -0.034814  0.049378  0.000173   
1        0.390542  1.386351 -0.179110  0.154334  0.300191  0.357683  0.012441   
2        1.394548  0.279114 -0.006633  0.022912  0.015378  0.104778 -0.096677   
3        0.773144  1.170151  0.103431  0.062477 -0.500634 -1.698183 -0.379848   
4        1.483493  1.444467  0.638430  0.016845  0.707832 -0.130665  0.006436   

movieId    8         9         10      ...    161084    161155    161594  \
0       -0.007822  0.013149  0.005328  ... -0.002265  0.000097  0.014752   
1        0.035352 -0.000300  2.227098  ... -0.001520  0.001430 -0.007687   
2        0.026333  0.008856  0.146107  ... -0.002164 -0.002974 -0.007573   
3       -0.154737 -0.109997  1.880732  ...  0.020092  0.006526  0.070166   
4        0.087133 -0.132831  0.277316  ... -0.000559  0.001697  0.004699   

movieId    161830    161918    161944    162376    16254

In [ ]:
import joblib
import numpy as np

def predecir(user_id, num_recommendations=10):
    """
    Predice las películas recomendadas para un usuario dado un user_id.

    :param user_id: El ID del usuario para el cual se desean hacer las recomendaciones.
    :param num_recommendations: Número de recomendaciones a generar (por defecto es 10).
    :return: Lista de IDs de películas recomendadas.
    """
    try:
        # Cargar el modelo entrenado de TruncatedSVD
        model_svd = joblib.load('models/modelo_svd.pkl')
        print("Modelo cargado exitosamente")

        # Aquí asumimos que tienes una matriz de características (user-item matrix) y estás utilizando el modelo para hacer recomendaciones.
        
        # Proyectamos al usuario en el espacio latente
        user_vector = model_svd.transform([[user_id]])  # Transformar el ID del usuario al espacio reducido
        
        # Ahora necesitamos calcular las similitudes entre el usuario y las películas.
        # Esto lo podemos hacer calculando la distancia entre el vector del usuario y las películas.
        
        # Vamos a obtener la matriz de características de las películas
        movie_features = model_svd.components_  # Esto da la matriz de características de las películas

        # Calcular similitudes utilizando el producto punto (esto devuelve los índices de las películas más cercanas)
        similarities = np.dot(movie_features, user_vector.T).flatten()

        # Ordenar las similitudes de mayor a menor y obtener los índices de las mejores recomendaciones
        recommended_movie_indices = similarities.argsort()[-num_recommendations:][::-1]

        # Retornar las recomendaciones (índices de las películas más cercanas)
        return recommended_movie_indices  # Retorna los índices de las películas recomendadas

    except Exception as e:
        print("Error en la predicción:", str(e))
        return None

if __name__ == "__main__":
    try:
        # Obtener user_id del usuario
        user_id = int(input("Introduzca un user id válido: "))
        
        # Obtener la lista de recomendaciones
        lista = predecir(user_id)
        
        if lista is not None:
            print(f"Lista de películas recomendadas para el usuario {user_id}: ")
            print(lista)
        else:
            print("No se pudieron generar recomendaciones.")
    
    except ValueError:
        print("Por favor, introduzca un user_id válido (un número entero).")


In [64]:
# Ejemplo de uso para obtener los 10 mejores 'movieId' para el usuario 1
user_id = 1
top_10_movie_ids = get_top_recommendations(user_id, 10)

# Imprimir las recomendaciones (solo los 'movieId')
print(f"Top 10 'movieId' recomendados para el usuario {user_id}:")
print(top_10_movie_ids)

Top 10 'movieId' recomendados para el usuario 1:
[296, 150, 590, 356, 457, 593, 480, 110, 380, 592]


Ahora, podemos guardar el modelo SVD para poder reutilizarlo de forma más eficiente

In [74]:
import joblib

# Guardar el modelo
joblib.dump(svd, '../modelo_svd.pkl')

['../modelo_svd.pkl']

### Conclusión

Hemos creado 3 modelos distintos de más a menos simple. Estos 3 modelos tienen diferencias pero pueden funcionar bien y, aligual que hace Netflix, podemos complementarlos.

Dependiendo de cada empresa, se podrá adaptar a sus datos y hacer cambios como por ejemplod arle más improtancia a una variable en concreto.

Estos modelos pueden combinarse y construir modelos híbridos y hacere diversos cambios, aunque lo ideal sería hacer pruebas y ver que funciona mejor para hacer cambios.